# 02 — Baselines zero-shot (decisão F0.5 do REPLAN)

Clona a voz do Pedro **sem treinar nada** em 3 sistemas license-clean e mede:
**A)** Chatterbox-Multilingual-**pt-br** (MIT, pack dedicado) · **B)** Pocket-TTS-pt
(Kyutai, CC-BY, CPU) · **C)** CSM-1B in-context (Apache). APIs verificadas 2026-06-10.

**Entrada:** 1-3 clipes limpos da voz (do piloto G0, ou grave 10s no celular pra começar).
**GPU:** T4 basta (Pocket roda em CPU). ⚠️ Rode a seção C em runtime SEPARADO
(pins de transformers conflitam com o chatterbox).

**Gate F0.5:** melhor (spk-sim × WER × escuta pt-BR-vs-pt-PT) vira o candidato #1 de finetune.
Se nenhum ≥0.60 spk-sim com pt-BR aceitável → direto pro finetune (notebooks 03/04).

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
GH_TOKEN = userdata.get('GH_TOKEN')
!git clone https://{GH_TOKEN}@github.com/pedrocormann/TTS-ptbr.git /content/TTS-ptbr 2>/dev/null || (cd /content/TTS-ptbr && git pull)
%cd /content/TTS-ptbr

# referência de voz: ~7-10s limpos (chatterbox trunca em 10s/6s; mais que isso é descartado)
# sugestão pronta (import ElevenLabs no Mac): data/raw/elevenlabs2024/segments/found_08_seg019.wav
# → copie pro Drive como ref_pedro.wav (ou escolha outro segmento limpo de 7-10s)
REF_WAV = '/content/drive/MyDrive/TTS-ptbr-data/ref_pedro.wav'   # ajuste o caminho
import pathlib; assert pathlib.Path(REF_WAV).exists(), 'suba um wav de referência no Drive'
import json
bench = [json.loads(l) for l in open('eval/benchmark_ptbr.jsonl', encoding='utf-8') if l.strip()]
print(len(bench), 'frases do benchmark congelado')

## A. Chatterbox-Multilingual-pt-br (pack dedicado — o multilingual amplo soa pt-PT, issue #281)

In [ ]:
!pip -q install chatterbox-tts soundfile jiwer librosa
from huggingface_hub import hf_hub_download
import shutil, pathlib

d = pathlib.Path('/content/ckpt_ptbr'); d.mkdir(exist_ok=True)
for f in ['t3_pt_br.safetensors', 's3gen_v3.pt', 'grapheme_mtl_merged_expanded_v1.json']:
    shutil.copy(hf_hub_download('ResembleAI/Chatterbox-Multilingual-pt-br', f), d / f)
for f in ['ve.pt', 'conds.pt', 'Cangjie5_TC.json']:    # voice encoder vem do repo base
    shutil.copy(hf_hub_download('ResembleAI/chatterbox', f), d / f)
if (d / 's3gen_v3.pt').exists():
    (d / 's3gen_v3.pt').rename(d / 's3gen.pt')          # from_local carrega 's3gen.pt'

import torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
model = ChatterboxMultilingualTTS.from_local(d, device='cuda', t3_model='t3_pt_br.safetensors')

out = pathlib.Path('gen_chatterbox'); out.mkdir(exist_ok=True)
for i, item in enumerate(bench):
    wav = model.generate(item['text'], language_id='pt', audio_prompt_path=REF_WAV,
                         exaggeration=0.5, cfg_weight=0.5, temperature=0.8)
    ta.save(str(out / f"{item.get('id', i):0>3}.wav"), wav, model.sr)   # 24kHz; watermark Perth embutida
print('✅ chatterbox pt-br ok →', out)
# expressivo: exaggeration~0.7 + cfg_weight~0.3 | referência com fala rápida: cfg_weight~0.3

## B. Pocket-TTS português (CPU, ~200ms; clone exige aceitar o gate em hf.co/kyutai/pocket-tts)

In [ ]:
!pip -q install pocket-tts
import os; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # gate aceito + token = clone liberado
# pré-exporta a voz (rápido depois) e gera nas DUAS variantes pt (destilada e 24l):
!pocket-tts export-voice {REF_WAV} /content/pedro_voice.safetensors --language portuguese
import json, pathlib, subprocess
for variant in ['portuguese', 'portuguese_24l']:
    out = pathlib.Path(f'gen_pocket_{variant}'); out.mkdir(exist_ok=True)
    for i, item in enumerate(bench):
        subprocess.run(['pocket-tts', 'generate', '--language', variant,
                        '--voice', '/content/pedro_voice.safetensors',
                        '--text', item['text'],
                        '--output-path', str(out / f"{item.get('id', i):0>3}.wav")], check=True)
    print('✅', variant, '→', out)
# escute: a voz default 'rafael' e o clone soam pt-BR ou pt-PT? ANOTE — é decisivo.

## C. CSM-1B in-context (⚠️ RUNTIME NOVO: pins conflitam) 

In [ ]:
# !pip -q install transformers==4.52.3 soundfile librosa
# import torch, soundfile as sf, librosa, json, pathlib
# from transformers import CsmForConditionalGeneration, AutoProcessor
# proc = AutoProcessor.from_pretrained('unsloth/csm-1b')
# csm = CsmForConditionalGeneration.from_pretrained('unsloth/csm-1b', torch_dtype=torch.bfloat16, device_map='cuda')
# ref_arr, _ = librosa.load(REF_WAV, sr=24000, mono=True)
# REF_TEXT = 'TRANSCRICAO EXATA DO SEU WAV DE REFERENCIA AQUI'
# out = pathlib.Path('gen_csm_zeroshot'); out.mkdir(exist_ok=True)
# bench = [json.loads(l) for l in open('eval/benchmark_ptbr.jsonl', encoding='utf-8') if l.strip()]
# for i, item in enumerate(bench):
#     conv = [{'role': '0', 'content': [{'type': 'text', 'text': REF_TEXT}, {'type': 'audio', 'path': ref_arr}]},
#             {'role': '0', 'content': [{'type': 'text', 'text': item['text']}]}]
#     inputs = proc.apply_chat_template(conv, tokenize=True, return_dict=True)
#     audio = csm.generate(**inputs.to('cuda'), max_new_tokens=375, output_audio=True)
#     sf.write(str(out / f"{item.get('id', i):0>3}.wav"), audio[0].to(torch.float32).cpu().numpy(), 24000)
# print('✅ csm zero-shot →', out)

## D. Eval comparativa (spk-sim + WER) → tabela de decisão

In [ ]:
!pip -q install faster-whisper==1.1.0 transformers
!mkdir -p ref_pedro && cp {REF_WAV} ref_pedro/
import pathlib
for system in ['gen_chatterbox', 'gen_pocket_portuguese', 'gen_pocket_portuguese_24l', 'gen_csm_zeroshot']:
    if not pathlib.Path(system).exists(): continue
    print(f'===== {system} =====')
    !python -m eval.wer_roundtrip --in-dir {system} --transcripts eval/benchmark_ptbr.jsonl --model medium --lang pt
    !python -m eval.speaker_sim --ref-dir ref_pedro --gen-dir {system}

## E. Decisão (preencher e levar pro REPLAN)

| Sistema | spk-sim | WER | Soa pt-BR? (ouvido) | Emoção controlável? | Veredito |
|---|---|---|---|---|---|
| Chatterbox-pt-br | | | | exaggeration/cfg | |
| Pocket-pt (6L) | | | | não | |
| Pocket-pt (24L) | | | | não | |
| CSM zero-shot | | | | contexto | |

→ Gate: melhor sistema = candidato #1; espelhe os `gen_*/` no Drive pra escuta
comparativa, e registre o resultado em `research/VIGIL-LOG.md` + REPLAN §F0.5.